# Chapter 8 — Diffusion Models and the U-Net Denoiser

Two chapters, two dead ends.

**Chapter 5** built autoencoders scored by per-pixel MSE, and they came out
**blurry**: when a model is unsure whether a detail belongs, squared error
rewards hedging, and a hedge is a smudge.

**Chapter 7** deleted the per-pixel loss and got sharpness from an adversary,
but paid for it with **instability**: no likelihood, no convergence criterion,
and a generator that can abandon half the data distribution without saying so.

Diffusion models get both, through a single reframing. Keep MSE, but stop
asking the network to produce a finished image in one shot. Instead, take a
nearly-clean image and ask it to remove *a little* noise. That prediction is so
tightly constrained that there is nothing to hedge about, so MSE stops causing
blur. Then do it again. And again, a few hundred times, starting from pure
static.

The training loop that results is as boring and stable as a supervised
classifier's. That boringness is why diffusion, rather than GANs, is behind
Stable Diffusion, Imagen, DALL·E 2, and Midjourney.

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | **Forward diffusion**: the fixed, closed-form process that destroys an image | `ylecun/mnist` |
| 2 | A **U-Net** denoiser with skip connections and timestep conditioning | — |
| 3 | Training it to predict the noise, with plain MSE | `ylecun/mnist` |
| 4 | **Reverse sampling**: pure static → a recognisable digit | — |
| 5 | **DDIM**: the same model, 10× fewer steps | — |
| 6 | **Class conditioning + classifier-free guidance**: how prompts work | `ylecun/mnist` |

**How each concept is presented**, the same three passes as earlier chapters:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** ~6 minutes on a GPU, covering two 40-epoch trainings at ~3 s/epoch,
with the rest spent in the sampling loops (a 400-step reverse pass is 400 forward
passes, which is the whole point of Module 5). On CPU, drop `EPOCHS` to 10 and
`T` to 200. MNIST is already cached from Chapter 2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset

keras.utils.set_random_seed(42)
rng = np.random.default_rng(seed=42)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none, so CPU only; reduce the knobs")

---
# Module 1 — Forward Diffusion: Destroying an Image on Purpose

## 1.1 The reframing

🧠 **The intuition.** Generating an image is hard. *Destroying* one is trivial:
stir in a little Gaussian noise, repeat a few hundred times, and you are left
with pure static. Nothing was learned. We wrote the destruction procedure
ourselves, and we know it exactly.

Now here is the whole idea. That destruction happened in small steps. If each
step was small enough, then **undoing one step is nearly as easy as doing it**,
since it is a mild denoising problem, exactly the kind you solved in Chapter 6,
Module 3. So: learn to undo *one* small step, then apply that skill several
hundred times in reverse, and you have walked all the way from static back to an
image that never existed.

We have converted "generate an image", which is impossibly hard, into "clean up
a slightly noisy image", which is easy, at the cost of doing the easy thing
hundreds of times. That trade is the entire field.

📐 **The math.** The **forward process** $q$ is a fixed Markov chain with no
learned parameters. Given a schedule $\beta_1, \dots, \beta_T$ of small
positive numbers:

$$ q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t; \; \sqrt{1 - \beta_t}\, x_{t-1}, \; \beta_t I\right) $$

Each step shrinks the current image slightly toward zero and adds a little
noise. The shrink factor $\sqrt{1 - \beta_t}$ is what keeps the variance from
growing without bound, chosen precisely so that if $x_0$ has unit variance, so
does every $x_t$.

**The closed form that makes this practical.** Iterating a Gaussian through
another Gaussian gives a Gaussian, so with
$\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$:

$$ \boxed{\;q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) I\right) \quad\Longleftrightarrow\quad x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon\;} $$

with $\varepsilon \sim \mathcal{N}(0, I)$. Read that boxed equation carefully,
because everything else depends on it: **to get a training example at timestep
900, you do not run 900 steps.** You draw one $\varepsilon$ and evaluate one
formula. Training at any noise level costs the same as training at any other.

(If it looks familiar, it should: it is the reparameterization trick from
Chapter 6, Module 2.1, a shifted and scaled standard normal.)

💻 **The code.** The whole forward process is a few lines of NumPy. There are
no weights here and nothing to train.


In [ ]:
T = 400                                       # <- knob: number of diffusion steps
BETA_START, BETA_END = 1e-4, 0.04

betas = np.linspace(BETA_START, BETA_END, T).astype("float32")
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas).astype("float32")     # ᾱ_t = Π α_s

print(f"T = {T} steps")
print(f"ᾱ at t=0:      {alpha_bar[0]:.4f}   -> signal kept: {np.sqrt(alpha_bar[0]):.3f}")
print(f"ᾱ at t={T//2}:    {alpha_bar[T // 2]:.4f}   -> signal kept: {np.sqrt(alpha_bar[T // 2]):.3f}")
print(f"ᾱ at t={T - 1}:   {alpha_bar[-1]:.6f} -> signal kept: {np.sqrt(alpha_bar[-1]):.3f}  (≈ pure noise)")


def forward_diffuse(x0, t, noise=None):
    """One-shot jump to timestep t:  x_t = √ᾱ_t · x₀ + √(1-ᾱ_t) · ε"""
    noise = rng.normal(size=x0.shape).astype("float32") if noise is None else noise
    ab = alpha_bar[t].reshape(-1, 1, 1, 1)
    return np.sqrt(ab) * x0 + np.sqrt(1.0 - ab) * noise

## 1.2 Load MNIST and watch a digit dissolve

We switch to MNIST for this chapter. Digits have clean, high-contrast strokes,
which makes it obvious at a glance whether the reverse process has produced
something structured or merely something textured.

Note the range: **[-1, 1]**, as in Chapter 7. This matters more here than it did
for the GAN. The forward process assumes the data is roughly zero-mean and
unit-variance, because that is what makes $x_T$ land on $\mathcal{N}(0, I)$,
the distribution we will start sampling from. Feed it $[0, 1]$ data and the
arithmetic stops matching, with no error raised to tell you.


In [ ]:
mnist = load_dataset("ylecun/mnist")


def split_to_arrays(split, n=None, image_col="image", label_col="label", seed=42):
    """Shuffle a HF split, optionally take the first n rows, return (images, labels)."""
    ds = split.shuffle(seed=seed)
    if n is not None:
        ds = ds.select(range(n))
    return np.stack([np.array(im) for im in ds[image_col]]), np.array(ds[label_col])


N_TRAIN = 30_000                                       # <- knob: up to 60_000
X_raw, y_train = split_to_arrays(mnist["train"], n=N_TRAIN)

X = (X_raw.astype("float32") / 127.5 - 1.0)[..., None]   # -> [-1, 1]
print("train:", X.shape, "| range:", X.min(), "to", X.max())

In [ ]:
steps_to_show = [0, 20, 50, 100, 150, 200, 260, 320, T - 1]
digit = X[:1]

fig, axes = plt.subplots(1, len(steps_to_show), figsize=(14, 2.1))
for ax, t in zip(axes, steps_to_show):
    ax.imshow(forward_diffuse(digit, np.array([t]))[0].squeeze(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t={t}\n√ᾱ={np.sqrt(alpha_bar[t]):.2f}", fontsize=8)
    ax.axis("off")
fig.suptitle("Forward diffusion: one image, nine noise levels, no training involved", y=1.14)
plt.tight_layout()
plt.show()

Notice **where** the information dies. The digit is still perfectly readable at
$t = 50$; by $t = 150$ it is a ghost; by $t = 260$ you would not bet on which
digit it was. The interesting learning happens in that middle band: early
steps are trivial to reverse, and late steps carry no signal to reverse toward.

## 1.3 The noise schedule is a design decision

🧠 **The intuition.** $\bar{\alpha}_t$ is a dial from "original image" to "pure
static", and the *shape* of its descent decides how the model's training effort
is distributed. A linear $\beta$ schedule, the original DDPM choice, destroys
the image faster than you would guess, spending a large share of its steps in
the regime where there is nothing left to learn from. The **cosine** schedule
(Nichol & Dhariwal, 2021) holds signal longer and degrades more gently at the
end, which measurably improves samples for small images like these.

📐 **The math.** The cosine schedule defines $\bar\alpha$ directly rather than
defining $\beta$:

$$ \bar{\alpha}_t = \frac{f(t)}{f(0)}, \qquad f(t) = \cos^2\!\left(\frac{t/T + s}{1 + s} \cdot \frac{\pi}{2}\right) $$

with a small offset $s = 0.008$ that prevents $\beta_t$ from vanishing at
$t = 0$. A useful quantity for comparing schedules is the **signal-to-noise
ratio**, $\mathrm{SNR}(t) = \bar{\alpha}_t / (1 - \bar{\alpha}_t)$, which is how
much image is left relative to how much noise.


In [ ]:
def cosine_alpha_bar(T, s=0.008):
    t = np.linspace(0, 1, T + 1)
    f = np.cos((t + s) / (1 + s) * np.pi / 2) ** 2
    return (f[1:] / f[0]).astype("float32")


alpha_bar_cos = cosine_alpha_bar(T)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(alpha_bar, label="linear β (used here)")
axes[0].plot(alpha_bar_cos, label="cosine")
axes[0].set_ylabel(r"$\bar{\alpha}_t$  (signal power retained)")
axes[0].set_title("How fast the image is destroyed")

axes[1].semilogy(alpha_bar / (1 - alpha_bar), label="linear β")
axes[1].semilogy(alpha_bar_cos / (1 - alpha_bar_cos), label="cosine")
axes[1].axhline(1.0, color="gray", linestyle="--", linewidth=1)
axes[1].set_ylabel("SNR (log scale)")
axes[1].set_title("Signal-to-noise ratio; below the line, noise dominates")

for ax in axes:
    ax.set_xlabel("timestep $t$"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

We keep the linear schedule for the rest of the notebook because it is the
original formulation and one fewer moving part, but swapping
`alpha_bar = alpha_bar_cos` in the cell above is a genuinely instructive
experiment once everything works.


---
# Module 2 — The U-Net Denoiser

## 2.1 Predict the noise, not the image

🧠 **The intuition.** We need a network that undoes one step of corruption. The
obvious design takes a noisy image and outputs the clean one. Diffusion models
do something subtly different and much better: they take the noisy image and
output **the noise that was added**.

Why does that help? Because at high $t$, "output the clean image" asks the
network to invent an entire digit from static, which is hard, and the hedging
problem from Chapter 5 comes straight back. "Output the noise" asks it to
identify which part of what it sees is *not* structure. It is the same
information (given $x_t$ and $\varepsilon$ you can solve for $x_0$
algebraically), but it is a far better-conditioned regression target: roughly
unit-variance at every timestep, with no dependence on how bright or dark the
underlying image is.

📐 **The math.** DDPM's remarkable result is that the full variational bound
(an ELBO over $T$ steps, structurally the same object as the VAE's in Chapter 6)
simplifies, after dropping some weighting terms, to this:

$$ \boxed{\;\mathcal{L}_{\text{simple}} = \mathbb{E}_{\,x_0,\; t \sim \mathcal{U}\{1,T\},\; \varepsilon \sim \mathcal{N}(0, I)} \Big[\; \big\|\, \varepsilon - \varepsilon_\theta(x_t, t) \,\big\|^2 \;\Big] \;}$$

Read what that says. Pick a training image. Pick a random timestep. Draw noise.
Make $x_t$ with the closed form from Module 1. Ask the network for the noise.
**Mean squared error.** That is the entire training objective of a diffusion
model: one line, no adversary, no KL term, no sampling loop during training.

🧠 **And note what it fixes.** MSE caused blur in Chapter 5 because the target
was ambiguous. Here the target $\varepsilon$ is *the exact tensor that was
added*, so there is a single right answer and no incentive to hedge toward an
average. Same loss function, completely different behaviour, purely because we
changed what is being predicted.

## 2.2 Telling the network what time it is

🧠 **The intuition.** One network must handle every noise level: at $t = 10$ it
should barely touch the image, at $t = 380$ it should ignore almost everything
it sees and reconstruct from priors. Those are wildly different behaviours, so
the network needs to know which one is being asked of it. We pass $t$ in as an
extra input.

But you cannot just feed the integer 380. A raw scalar is a poor input to a
neural network: the difference between $t = 100$ and $t = 101$ matters, and so
does the difference between $t = 100$ and $t = 300$, at completely different
scales. So we expand $t$ into a vector of sines and cosines at many
frequencies: fast components distinguish neighbouring timesteps, slow ones
capture the broad regime. It is a "ruler with both fine and coarse gradations".

📐 **The math.** For embedding dimension $d$, with $k = 0, \dots, d/2 - 1$:

$$ \text{emb}(t)_k = \sin\!\left(\frac{t}{10000^{\,2k/d}}\right), \qquad \text{emb}(t)_{k + d/2} = \cos\!\left(\frac{t}{10000^{\,2k/d}}\right) $$

This is *exactly* the positional encoding from "Attention Is All You Need", the
same construction used to tell a Transformer where a token sits in a
sentence, here used to tell a U-Net where an image sits on the noise ladder.

💻 **The code.**


In [ ]:
class SinusoidalEmbedding(layers.Layer):
    """Turn an integer timestep into a d-dimensional vector of sines and cosines."""

    def __init__(self, dim, max_period=10000.0, **kwargs):
        super().__init__(**kwargs)
        self.dim, self.max_period = dim, max_period

    def call(self, t):
        half = self.dim // 2
        freqs = tf.exp(-tf.math.log(self.max_period) * tf.range(half, dtype="float32") / half)
        args = tf.cast(t, "float32")[:, None] * freqs[None, :]      # (B, half)
        return tf.concat([tf.sin(args), tf.cos(args)], axis=-1)     # (B, dim)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.dim)


# See what it produces: each row is one timestep's fingerprint.
probe = SinusoidalEmbedding(64)(tf.constant([0, 10, 50, 100, 200, 399]))
plt.figure(figsize=(9, 2.4))
plt.imshow(probe.numpy(), aspect="auto", cmap="RdBu_r")
plt.yticks(range(6), ["t=0", "t=10", "t=50", "t=100", "t=200", "t=399"], fontsize=8)
plt.xlabel("embedding dimension"); plt.colorbar(label="value")
plt.title("Sinusoidal timestep embeddings: nearby timesteps look similar, distant ones don't")
plt.tight_layout()
plt.show()

## 2.3 The U-Net

🧠 **The intuition.** The denoiser's output must be the same size as its input,
a full 28×28 noise map rather than a label. That immediately rules out the plain
CNN shape you know, and it also creates a tension:

- To decide *what* the noise is, the network needs **global context**. Is this
  bright pixel a stroke or a speckle? You cannot tell from a 3×3 neighbourhood;
  you have to know the digit is probably a 7 and that a stroke belongs there.
  Global context requires downsampling.
- But downsampling destroys the **precise spatial detail** needed to output a
  per-pixel answer. The bottleneck knows *what*, and has forgotten *where*.

The **U-Net** (Ronneberger et al., 2015, for biomedical segmentation) resolves
this with **skip connections**: at each resolution on the way down, stash a copy
of the feature map; on the way back up, concatenate it to the upsampled
features. The decoder gets the "what" from the bottleneck and the "where"
straight from the encoder, over a shortcut that never passed through the
squeeze. Draw the shapes and it looks like a **U**.

You have already built both halves. The contracting path is Chapter 5's
encoder; the expanding path is Chapter 5's decoder. The skip connections are
Chapter 3's residual shortcuts, wired between distant layers instead of adjacent
ones. A U-Net is not a new architecture so much as the two things you know,
joined.

📐 **The shape journey.**

```
 28×28×64  ──── skip ────────────────────────────────►  concat → 28×28×128 → out 28×28×1
     │ stride-2 conv                                          ▲ transposed conv
 14×14×128 ──── skip ──────────────►  concat → 14×14×256      │
     │ stride-2 conv                       ▲ transposed conv
  7×7×256  ──── bottleneck: 2 residual blocks ──────────┘
```

💻 **The code.** Three ingredients per block: `GroupNormalization` (a batch-size
independent alternative to BatchNorm, which matters because our "batch" during
sampling might be 8 images), the `swish` activation, and the timestep vector
projected to this block's channel count and **added as a per-channel bias**.
That last line is how $t$ actually influences the computation: it shifts every
feature map according to where we are on the noise ladder.


In [ ]:
def residual_block(x, t_emb, filters, groups=8):
    """Conv → norm → swish, twice, with the timestep injected and an identity shortcut."""
    shortcut = x if x.shape[-1] == filters else layers.Conv2D(filters, 1)(x)

    h = layers.GroupNormalization(groups=groups)(x)
    h = layers.Activation("swish")(h)
    h = layers.Conv2D(filters, 3, padding="same")(h)

    # inject t: project the embedding to `filters` numbers and add one per channel
    t_bias = layers.Dense(filters)(layers.Activation("swish")(t_emb))
    h = layers.Add()([h, layers.Reshape((1, 1, filters))(t_bias)])

    h = layers.GroupNormalization(groups=groups)(h)
    h = layers.Activation("swish")(h)
    h = layers.Conv2D(filters, 3, padding="same")(h)

    return layers.Add()([h, shortcut])          # the residual connection from Chapter 3


def build_unet(base=64, t_dim=128, num_classes=None, name="unet"):
    x_in = keras.Input(shape=(28, 28, 1), name="noisy_image")
    t_in = keras.Input(shape=(), dtype="int32", name="timestep")
    inputs = [x_in, t_in]

    # --- conditioning vector: timestep (+ optional class label) --------------
    t_emb = SinusoidalEmbedding(t_dim)(t_in)
    t_emb = layers.Dense(t_dim * 2, activation="swish")(t_emb)
    t_emb = layers.Dense(t_dim * 2)(t_emb)

    if num_classes is not None:                       # used in Module 6
        y_in = keras.Input(shape=(), dtype="int32", name="label")
        inputs.append(y_in)
        # num_classes + 1: the extra index is the "no label given" token
        y_emb = layers.Embedding(num_classes + 1, t_dim * 2)(y_in)
        t_emb = layers.Add()([t_emb, y_emb])

    # --- contracting path (Chapter 5's encoder) -----------------------------
    h = layers.Conv2D(base, 3, padding="same")(x_in)                   # 28x28x64
    skip1 = residual_block(h, t_emb, base)
    h = layers.Conv2D(base * 2, 3, strides=2, padding="same")(skip1)   # 14x14x128
    skip2 = residual_block(h, t_emb, base * 2)
    h = layers.Conv2D(base * 4, 3, strides=2, padding="same")(skip2)   #  7x7x256

    # --- bottleneck ---------------------------------------------------------
    h = residual_block(h, t_emb, base * 4)
    h = residual_block(h, t_emb, base * 4)

    # --- expanding path (Chapter 5's decoder) + skips -----------------------
    h = layers.Conv2DTranspose(base * 2, 4, strides=2, padding="same")(h)   # 14x14x128
    h = layers.Concatenate()([h, skip2])                                    # <- the U
    h = residual_block(h, t_emb, base * 2)

    h = layers.Conv2DTranspose(base, 4, strides=2, padding="same")(h)       # 28x28x64
    h = layers.Concatenate()([h, skip1])                                    # <- the U
    h = residual_block(h, t_emb, base)

    h = layers.GroupNormalization(groups=8)(h)
    h = layers.Activation("swish")(h)
    noise_pred = layers.Conv2D(1, 3, padding="same", name="predicted_noise")(h)

    return keras.Model(inputs, noise_pred, name=name)


unet = build_unet()
print(f"U-Net parameters: {unet.count_params():,}")
unet.summary(line_length=100)

---
# Module 3 — Training

🧠 **The intuition.** Build the training set on the fly. For every image in
every epoch, roll a random timestep, roll a fresh noise tensor, corrupt the
image to that level, and hand the network the pair `(noisy image, timestep)`
with the noise as the label. Because $t$ is re-rolled every epoch, the same
image teaches the network about a different noise level each time round.

💻 **The code.** A `tf.data` pipeline does the corruption, so the "labels" are
manufactured inside the input pipeline and `model.fit` becomes an ordinary
supervised call. Compare this cell to Chapter 7's two-tape adversarial loop,
because that contrast is the practical argument for diffusion.


In [ ]:
alpha_bar_tf = tf.constant(alpha_bar)


def make_training_pair(x0):
    """x0 -> ((x_t, t), ε): one random timestep, one fresh noise draw."""
    t = tf.random.uniform((), minval=0, maxval=T, dtype=tf.int32)
    eps = tf.random.normal(tf.shape(x0))
    ab = tf.gather(alpha_bar_tf, t)
    x_t = tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * eps      # the boxed formula from Module 1
    return (x_t, t), eps


BATCH = 128
train_ds = (tf.data.Dataset.from_tensor_slices(X)
            .shuffle(len(X), seed=42, reshuffle_each_iteration=True)
            .map(make_training_pair, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH, drop_remainder=True)
            .prefetch(tf.data.AUTOTUNE))

(demo_x, demo_t), demo_eps = next(iter(train_ds))
print("one batch:", demo_x.shape, "| timesteps:", demo_t.numpy()[:8], "| target:", demo_eps.shape)

In [ ]:
# Keras enables XLA automatically on GPU here, which is worth ~3x on this model.
# If your setup hits an XLA compilation error, pass jit_compile=False.
unet.compile(optimizer=keras.optimizers.Adam(2e-4), loss="mse")

EPOCHS = 40                                           # <- knob: 15 gives blurry-but-real digits
hist = unet.fit(train_ds, epochs=EPOCHS, verbose=2)

plt.plot(hist.history["loss"], marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE on predicted noise")
plt.title("Diffusion training: as dull as a supervised classifier, and that is the point")
plt.grid(True, alpha=0.3)
plt.show()

⚠️ **Do not read too much into this number.** The loss averages over all
timesteps, and the difficulty varies enormously across them: at $t \approx 0$
the input is almost clean and the noise is almost impossible to pin down, while
at large $t$ the input *is* the noise and prediction is nearly free. A loss
around 0.03 is not "3% error" in any meaningful sense. Unlike a GAN's, this
curve at least moves monotonically, but sample quality is still something you
check by looking.

## 3.1 Does it actually denoise?

Before the full reverse loop, a direct test of what we trained: corrupt a real
digit to some $t$, ask for the noise, and subtract it. Rearranging the boxed
formula gives the model's implied guess at the clean image:

$$ \hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t}\; \varepsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}} $$


In [ ]:
def predict_x0(x_t, t, model=unet, extra_inputs=()):
    """Estimate the clean image in one jump from x_t."""
    t_batch = np.full(len(x_t), t, dtype="int32")
    eps_pred = model([x_t, t_batch, *extra_inputs], training=False).numpy()
    ab = alpha_bar[t]
    return (x_t - np.sqrt(1 - ab) * eps_pred) / np.sqrt(ab)


test_digits = X[:6]
fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for col in range(6):
    axes[0, col].imshow(test_digits[col].squeeze(), cmap="gray", vmin=-1, vmax=1)

t_probe = 200                                          # <- knob: try 50, 200, 350
noisy = forward_diffuse(test_digits, np.full(6, t_probe))
guess = predict_x0(noisy, t_probe)

for col in range(6):
    axes[1, col].imshow(noisy[col].squeeze(), cmap="gray", vmin=-1, vmax=1)
    axes[2, col].imshow(np.clip(guess[col].squeeze(), -1, 1), cmap="gray", vmin=-1, vmax=1)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
for row, tag in enumerate(["original", f"noised to t={t_probe}", "model's $\\hat{x}_0$"]):
    axes[row, 0].set_ylabel(tag, fontsize=9)
plt.tight_layout()
plt.show()

The one-jump estimate is **blurry**, and that blur is Chapter 5's ghost, back
again for exactly the same reason. Asked to leap from $t = 200$ to a finished
image in one move, the model is genuinely uncertain about the details, and MSE
tells it to hedge. This is the single best argument for why sampling proceeds
in small steps: each step's uncertainty is tiny, so each step's hedge is
negligible.


---
# Module 4 — Reverse Sampling: Static → Digit

🧠 **The intuition.** Start with pure static, a tensor of random numbers with no
image inside it. Tell the model it is at $t = 399$ and ask what noise it sees.
Remove *a fraction* of what it names, add back a smaller amount of fresh noise,
and call the result $t = 398$. Repeat down to zero.

Two questions people always ask:

**Why remove only a fraction?** Because the model's estimate is uncertain, and
subtracting all of it commits hard to a blurry average (you just saw this in
3.1). Removing one step's worth keeps every commitment small.

**Why add noise *back* in?** It seems perverse. But it is what makes this a
*sampler* rather than an optimizer. Without the injected randomness, the
procedure is deterministic given $x_T$ and collapses toward the mean of the
data; the injected noise is what lets the trajectory commit to *one* digit
rather than an average of all of them, and it is what makes the ten digits come
out in roughly equal proportion. Diffusion's freedom from mode collapse is
purchased here.

📐 **The math.** The reverse step, from Ho et al. (2020):

$$ x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \varepsilon_\theta(x_t, t)\right) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I) $$

The bracket is the posterior mean of $x_{t-1}$ given $x_t$ and the predicted
noise; $\sigma_t z$ is the injected randomness, set to zero on the final step.
We use the posterior variance
$\sigma_t^2 = \tilde{\beta}_t = \beta_t \dfrac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}$,
which is slightly tighter than the simpler choice $\sigma_t^2 = \beta_t$.

💻 **The code.**


In [ ]:
alpha_bar_prev = np.append(1.0, alpha_bar[:-1]).astype("float32")
posterior_var = (betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar)).astype("float32")


def ddpm_sample(n=8, model=unet, extra_inputs=(), record_every=None, seed=0):
    """Ancestral sampling: pure noise at t=T-1, walk down to t=0."""
    gen = np.random.default_rng(seed)
    x = gen.normal(size=(n, 28, 28, 1)).astype("float32")     # x_T ~ N(0, I)
    trajectory = {}

    for t in range(T - 1, -1, -1):
        t_batch = np.full(n, t, dtype="int32")
        eps_pred = model([x, t_batch, *extra_inputs], training=False).numpy()

        mean = (x - betas[t] / np.sqrt(1 - alpha_bar[t]) * eps_pred) / np.sqrt(alphas[t])
        if t > 0:
            z = gen.normal(size=x.shape).astype("float32")
            x = mean + np.sqrt(posterior_var[t]) * z          # inject noise...
        else:
            x = mean                                          # ...except on the last step

        if record_every and (t % record_every == 0 or t == 0):
            trajectory[t] = x.copy()

    return x, trajectory


samples, trajectory = ddpm_sample(n=8, record_every=50, seed=3)
print("sampled", samples.shape, "| trajectory snapshots at t =", sorted(trajectory, reverse=True))

In [ ]:
snaps = sorted(trajectory, reverse=True)
fig, axes = plt.subplots(4, len(snaps), figsize=(1.15 * len(snaps), 5))
for col, t in enumerate(snaps):
    for row in range(4):
        axes[row, col].imshow(trajectory[t][row].squeeze(), cmap="gray", vmin=-1, vmax=1)
        axes[row, col].axis("off")
    axes[0, col].set_title(f"t={t}", fontsize=8)
fig.suptitle("The reverse trajectory: four samples walking from static to digits", y=1.03)
plt.tight_layout()
plt.show()

Watch *when* each image decides what it is going to be. For most of the descent
there is only a vague blob of density. Somewhere around $t \approx 150$–$100$
the global structure locks in, the sample commits to a digit identity, and
the remaining steps only sharpen strokes and clean the background. That
coarse-to-fine ordering is emergent; nobody programmed it. It falls out of the
noise schedule, because low-frequency structure survives corruption longer than
high-frequency detail does, and so is the first thing recoverable on the way
back.


In [ ]:
def show_grid(images, title, rows=3, cols=8, vmin=-1, vmax=1):
    fig, axes = plt.subplots(rows, cols, figsize=(1.25 * cols, 1.35 * rows))
    for ax, img in zip(axes.flat, images):
        ax.imshow(np.clip(img.squeeze(), vmin, vmax), cmap="gray", vmin=vmin, vmax=vmax)
        ax.axis("off")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


many, _ = ddpm_sample(n=24, seed=11)
show_grid(many, f"24 digits generated from pure noise, {T} reverse steps each")

---
# Module 5 — DDIM: The Same Model, Ten Times Faster

🧠 **The intuition.** That grid took $T = 400$ forward passes *per image*. A GAN
needs one. This is diffusion's real cost, and it is why the field spent years on
faster samplers.

**DDIM** (Song et al., 2020) is the key realisation: the training objective
never actually required the reverse process to be the exact Markov chain the
forward process defined. The network learned "given a noisy image at level $t$,
what is the noise", a question you can ask at *any* $t$, in any order. So we
can skip: go straight from $t = 399$ to $t = 379$ to $t = 359$, using the same
weights, with no retraining whatsoever.

Drop the injected randomness as well and the sampler becomes **deterministic**:
the same starting noise always yields the same image, which makes $x_T$ a true
latent code for the generated image, the interpolatable latent space from
Chapter 5, recovered.

📐 **The math.** With a subsequence $\tau_1 < \dots < \tau_S$ of timesteps, each
DDIM step estimates $x_0$ and then re-noises it to the *next* level down:

$$ \hat{x}_0 = \frac{x_{\tau_i} - \sqrt{1 - \bar{\alpha}_{\tau_i}}\, \varepsilon_\theta(x_{\tau_i}, \tau_i)}{\sqrt{\bar{\alpha}_{\tau_i}}}, \qquad x_{\tau_{i-1}} = \sqrt{\bar{\alpha}_{\tau_{i-1}}}\, \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{\tau_{i-1}}}\, \varepsilon_\theta(x_{\tau_i}, \tau_i) $$

Both pieces are already familiar: the first is the $\hat{x}_0$ estimate from
Module 3.1, the second is Module 1's forward formula, reused to step *down*
instead of up. (This is the $\eta = 0$ case of the DDIM family; $\eta = 1$
recovers DDPM.)

💻 **The code.**


In [ ]:
def ddim_sample(n=8, steps=40, model=unet, extra_inputs=(), seed=0, x_T=None):
    """Deterministic sampling over a subsequence of timesteps."""
    gen = np.random.default_rng(seed)
    x = gen.normal(size=(n, 28, 28, 1)).astype("float32") if x_T is None else x_T.copy()

    schedule = np.linspace(T - 1, 0, steps).round().astype("int32")
    for i, t in enumerate(schedule):
        t_batch = np.full(n, t, dtype="int32")
        eps_pred = model([x, t_batch, *extra_inputs], training=False).numpy()

        x0_hat = (x - np.sqrt(1 - alpha_bar[t]) * eps_pred) / np.sqrt(alpha_bar[t])
        ab_prev = alpha_bar[schedule[i + 1]] if i + 1 < len(schedule) else 1.0
        x = np.sqrt(ab_prev) * x0_hat + np.sqrt(1 - ab_prev) * eps_pred   # re-noise to the next level

    return x


import time

for steps in (10, 25, 50):                             # <- knob
    t0 = time.time()
    imgs = ddim_sample(n=8, steps=steps, seed=5)
    show_grid(imgs, f"DDIM, {steps} steps  ({time.time() - t0:.1f} s for 8 images)", rows=1, cols=8)

At 10 steps the digits are recognisable but coarse; by 25–50 they are close to
the full 400-step DDPM output, at a tenth of the cost. This is the trade every
production image generator exposes as a "steps" or "quality" slider, and
modern samplers (DPM-Solver++ among others) push acceptable quality down to
4–10 steps by treating the reverse process as an ODE and applying a better
numerical integrator to it.

## 5.1 A latent space, recovered

Because DDIM is deterministic, the initial noise $x_T$ *is* a latent code.
Interpolate between two noise tensors and you interpolate between two digits,
the Chapter 5 experiment, now on a model that produces sharp images. We use
spherical interpolation, for the same reason as in Chapter 7: the midpoint of a
straight chord between two Gaussian samples is off-distribution.


In [ ]:
def slerp(z1, z2, steps):
    a = np.linspace(0, 1, steps).reshape(-1, *([1] * z1.ndim))
    n1, n2 = z1 / np.linalg.norm(z1), z2 / np.linalg.norm(z2)
    omega = np.arccos(np.clip(np.sum(n1 * n2), -1, 1))
    return (np.sin((1 - a) * omega) * z1 + np.sin(a * omega) * z2) / np.sin(omega)


noise_a = rng.normal(size=(28, 28, 1)).astype("float32")
noise_b = rng.normal(size=(28, 28, 1)).astype("float32")
path = slerp(noise_a, noise_b, 8).astype("float32")

show_grid(ddim_sample(n=8, steps=50, x_T=path), "Interpolating in noise space with a deterministic sampler",
          rows=1, cols=8)

---
# Module 6 — Conditioning: How a Prompt Steers a Diffusion Model

🧠 **The intuition.** Everything so far samples from "all of MNIST". Useful
generation means asking for something specific, and the mechanism, remarkably,
is the one you already built. The timestep is fed to the U-Net as a vector
added into every block. A class label can ride in the exact same way: embed it,
add it to the timestep vector, done. That is the entire architectural change,
and it is the `num_classes` branch already sitting in `build_unet`.

**Classifier-free guidance** is the second half, and it is the trick that made
text-to-image work as well as it does. Train *one* model that sometimes sees
the label and sometimes sees a "no label given" token, by dropping the label at
random 10% of the time. At sampling time you can then run the same network
twice, once conditioned and once not, and **extrapolate away from the
unconditional prediction**:

> "Give me the direction that makes this more like a 7 *and less like a generic
> digit*, and then take an exaggerated step that way."

Turn the exaggeration up and samples become more prototypical, more obedient to
the label, and less diverse. That dial is exactly the `guidance_scale` /
`cfg` slider in every Stable Diffusion interface.

📐 **The math.** With $\varnothing$ denoting the null label:

$$ \tilde{\varepsilon}_\theta(x_t, t, y) = \varepsilon_\theta(x_t, t, \varnothing) + w \cdot \big[\varepsilon_\theta(x_t, t, y) - \varepsilon_\theta(x_t, t, \varnothing)\big] $$

$w = 1$ is ordinary conditional sampling; $w = 0$ ignores the label entirely;
$w > 1$ is guidance, and typical production values are 3–10.

💻 **The code.** Same architecture, same training loop, and the pipeline just
emits a label alongside, dropped to the null token 10% of the time.


In [ ]:
NUM_CLASSES = 10
NULL_TOKEN = NUM_CLASSES                               # index 10 = "no label given"
LABEL_DROP = 0.1                                       # <- knob

cond_unet = build_unet(num_classes=NUM_CLASSES, name="conditional_unet")
print(f"conditional U-Net parameters: {cond_unet.count_params():,}")


def make_conditional_pair(x0, y):
    """x0, y -> ((x_t, t, y_or_null), ε)"""
    t = tf.random.uniform((), minval=0, maxval=T, dtype=tf.int32)
    eps = tf.random.normal(tf.shape(x0))
    ab = tf.gather(alpha_bar_tf, t)
    x_t = tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * eps

    drop = tf.random.uniform(()) < LABEL_DROP          # sometimes hide the label...
    y_out = tf.where(drop, tf.cast(NULL_TOKEN, tf.int32), tf.cast(y, tf.int32))
    return (x_t, t, y_out), eps


cond_ds = (tf.data.Dataset.from_tensor_slices((X, y_train))
           .shuffle(len(X), seed=42, reshuffle_each_iteration=True)
           .map(make_conditional_pair, num_parallel_calls=tf.data.AUTOTUNE)
           .batch(BATCH, drop_remainder=True)
           .prefetch(tf.data.AUTOTUNE))

cond_unet.compile(optimizer=keras.optimizers.Adam(2e-4), loss="mse")
cond_hist = cond_unet.fit(cond_ds, epochs=EPOCHS, verbose=2)

In [ ]:
def guided_sample(labels, steps=50, guidance=3.0, seed=0):
    """DDIM sampling with classifier-free guidance."""
    n = len(labels)
    gen = np.random.default_rng(seed)
    x = gen.normal(size=(n, 28, 28, 1)).astype("float32")

    y_cond = np.asarray(labels, dtype="int32")
    y_null = np.full(n, NULL_TOKEN, dtype="int32")
    schedule = np.linspace(T - 1, 0, steps).round().astype("int32")

    for i, t in enumerate(schedule):
        t_batch = np.full(n, t, dtype="int32")
        eps_cond = cond_unet([x, t_batch, y_cond], training=False).numpy()
        eps_null = cond_unet([x, t_batch, y_null], training=False).numpy()
        eps = eps_null + guidance * (eps_cond - eps_null)     # <- the guidance formula

        x0_hat = (x - np.sqrt(1 - alpha_bar[t]) * eps) / np.sqrt(alpha_bar[t])
        ab_prev = alpha_bar[schedule[i + 1]] if i + 1 < len(schedule) else 1.0
        x = np.sqrt(ab_prev) * x0_hat + np.sqrt(1 - ab_prev) * eps

    return x


requested = np.repeat(np.arange(10), 3)                # three of each digit, on demand
show_grid(guided_sample(requested, steps=50, guidance=3.0, seed=1),
          "Generated on request: three samples of each digit 0–9 (guidance = 3.0)",
          rows=3, cols=10)

## 6.1 Turning the guidance dial

Same starting noise, same label, four values of $w$. This is the single most
useful plot for building intuition about a slider you will otherwise be
adjusting by superstition.

In [ ]:
fixed_labels = np.arange(10)
fig, axes = plt.subplots(4, 10, figsize=(12, 5.2))
for row, w in enumerate([0.0, 1.0, 3.0, 8.0]):
    imgs = guided_sample(fixed_labels, steps=40, guidance=w, seed=99)
    for col in range(10):
        axes[row, col].imshow(np.clip(imgs[col].squeeze(), -1, 1), cmap="gray", vmin=-1, vmax=1)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    axes[row, 0].set_ylabel(f"w={w}", fontsize=9)
for col in range(10):
    axes[0, col].set_title(str(col), fontsize=9)
fig.suptitle("Classifier-free guidance strength (columns are the requested digit)", y=1.02)
plt.tight_layout()
plt.show()

Read the rows:

| $w$ | What you get |
|---|---|
| **0** | The label is ignored entirely, so you get random digits, because this is literally the unconditional model |
| **1** | Plain conditional sampling. It mostly obeys, with natural variety in stroke weight and slant |
| **3** | Obedient and clean. This is the sweet spot for most models |
| **8** | Over-driven. Digits become exaggerated, thick, prototypical; contrast blows out and diversity collapses, the same over-saturated, over-literal look you get from cranking CFG too high in Stable Diffusion |

Guidance trades **diversity for fidelity to the condition**. There is no free
lunch on that dial, in this notebook or in a production text-to-image model.


---
# Wrap-Up: How This Becomes Stable Diffusion

| You built | The transferable lesson |
|---|---|
| Forward diffusion + closed form | Destruction is free and exact; $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\varepsilon$ lets you train any noise level in O(1) |
| Noise-prediction objective | Plain MSE stops causing blur once the target is the noise, because the target is now unambiguous |
| Sinusoidal timestep embedding | The Transformer's positional encoding, telling a U-Net where it is on the noise ladder |
| U-Net with skip connections | Chapter 5's encoder-decoder + Chapter 3's shortcuts; bottleneck supplies *what*, skips supply *where* |
| DDPM ancestral sampling | Injected noise is what makes it a sampler, and what makes mode collapse structurally impossible |
| DDIM | The reverse chain need not match the forward one; skip steps, go deterministic, recover a latent space |
| Classifier-free guidance | Train with the label dropped 10% of the time; extrapolate away from unconditional at sample time |

**The gap to a production text-to-image model is smaller than it looks.** Three
changes to what you just built:

1. **Diffuse in a latent space, not pixel space.** A 512×512 image is 786,432
   numbers, and running 50 U-Net passes over that is ruinous. **Latent
   diffusion** first trains an autoencoder (Chapter 5, Module 2, scaled up) to
   compress images ~48× into a 64×64×4 latent grid, then runs this entire
   notebook *inside that latent space*. The "Stable" in Stable Diffusion is
   largely this. Chapters 5 and 8 are the two halves of it.
2. **Condition on text instead of a class index.** Replace
   `layers.Embedding(11, 256)` with a frozen text encoder (CLIP or T5) and
   inject its output through **cross-attention** in each U-Net block rather than
   a simple additive bias. Classifier-free guidance carries over unchanged,
   since the null token becomes the empty string, which is exactly what a
   "negative prompt" field manipulates.
3. **Scale.** Bigger U-Net, attention layers at the lower resolutions, hundreds
   of millions of images. No new ideas, just the ones in this notebook, larger.

**Where the generative chapters have taken you.** Chapters 5 and 6 gave you
latent spaces, the encoder-decoder, and the first model you could actually
sample from. Chapter 7 gave you adversarial training and the reasons the field
moved on from it. This chapter gave you the denoising formulation that now
underpins essentially all image, video, and audio generation. The natural next
step is **attention**, the mechanism that carries text conditioning into the
U-Net, and the one architectural idea in modern generative AI you have not yet
built from scratch.


In [ ]:
final = ddim_sample(n=24, steps=50, seed=2024)
show_grid(final, "Twenty-four handwritten digits that were never written", rows=3, cols=8)